# 01. Qwen으로 Val 칸 판정 (A100, 15~30분)

**목적**: Val의 라벨 오류 칸을 찾는다. 모델 예측은 보지 않고, 대화를 읽은 판정관(Qwen3-8B)의 답만 쓴다. Train(02)에도 똑같이 적용한다.

**질문**(칸 하나마다): "신고자 쪽이 이 증상을 말했나?"
A 호소 / B 대원이 묻기만 함 / C 언급 없음 / D 신고자가 부정 → 네 확률(`p_호소`, `p_대원질문`, `p_없음`, `p_부정`)을 쓴다.

**처리 기준** (`config.JudgeConfig`, 실제 적용은 03)
| 원래 라벨 | Qwen 확률 | 처리 |
|---|---|---|
| 있음 | p_호소 ≥ 0.5 | 그대로 |
| 있음 | p_없음 + p_부정 ≥ 0.9 | '없음'으로 수정 |
| 있음 | 그 사이 (대원만 물음 등) | 삭제 |
| 없음 | p_호소 ≥ 0.9 | '있음'으로 수정 |
| 없음 | 0.5 ≤ p_호소 < 0.9 | 삭제 |
| 없음 | p_호소 < 0.5 | 그대로 |

구토·오심은 하나로 합쳐 묻는다(서로 헷갈린 것은 오류가 아님).

| 단계 | 하는 일 | 쓰는 함수 |
|---|---|---|
| 0 | 준비, Qwen 불러오기 | `judge_llm.load_judge` |
| 1 | 판정할 칸 만들기 | `cells.build_cells` |
| 2 | **Qwen 판정** (2,000칸마다 저장, 끊기면 다시 실행) | `cells.judge_cells` |
| 3 | 판정표 저장 + 처리 표시 + 증상별 분포표 | `cells.save_judged`, `cleaning.decide_actions`, `cleaning.action_summary` |
| 4 | **신뢰도 1차 확인**: 대회 때 Claude 판정과 일치율 | `evaluate.agree_with_claude` |
| 5 | **검수 파일** (없음으로 수정 40 + 있음으로 수정 40 + 삭제 30 + 그대로 40) | `cleaning.review_sample`, `cleaning.write_review_txt` |

결과: `judge/val/`, `results/01_val/`. 원본 데이터는 건드리지 않는다.

## 0. 준비

In [ ]:
# Qwen3는 transformers 4.51 이상이 필요하다. 설치 후 '세션 다시 시작' 메시지가 나오면 다시 시작하고 위 셀부터 실행
!pip -q install -U "transformers>=4.51" accelerate

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# src/*.py를 고친 뒤에는 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
import pandas as pd
import torch
from src import config
from src.data import load_labels
from src.cells import build_cells, judge_cells, save_judged
from src.cleaning import decide_actions, action_summary, review_sample, write_review_txt
from src.evaluate import agree_with_claude
from src.judge_llm import load_judge

jcfg = config.JudgeConfig()     # 메모리가 작은 GPU(T4 등)면 config.JudgeConfig(load_in_4bit=True, batch_size=4)
OUT = f"{config.RESULTS_DIR}/01_val"
os.makedirs(OUT, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(GPU 런타임으로 바꾸기)")

In [ ]:
# 처음엔 모델 내려받기에 몇 분 걸린다(약 16GB)
tok, llm = load_judge(jcfg.model_name, jcfg.load_in_4bit)

## 1. 판정할 칸

In [ ]:
_, va_df = load_labels()
va_cells = build_cells(va_df)

## 2. Qwen 판정
블록마다 걸린 시간과 남은 시간을 출력한다. 끊기면 0단계부터 다시 실행하면 저장된 블록은 건너뛴다.

In [ ]:
import time
t0 = time.time()
va_probs, done = judge_cells(va_cells, tok, llm, f"{config.JUDGE_DIR}/val", jcfg)
val_min = (time.time() - t0) / 60
print(f"이번 실행 {val_min:.1f}분")

## 3. 판정표 저장 + 처리 표시 + 증상별 분포표

In [ ]:
assert done, "판정이 덜 끝남. 위 셀을 다시 실행"
judged = save_judged(va_cells, va_probs, f"{config.JUDGE_DIR}/val_judged.csv")
judged = decide_actions(judged, jcfg)
summ = action_summary(judged, va_df, jcfg.warn_min_pos_val, jcfg)
summ.to_csv(f"{OUT}/처리_분포표.csv", encoding="utf-8-sig")
display(summ)

## 4. 신뢰도 1차 확인: 대회 때 Claude 판정과 비교
대회 때 Claude가 Val 약 4천 칸을 직접 읽고 '호소/없음'을 판정했다(`val_claude_judge.xlsx`, 없으면 건너뜀).
**'Qwen 없음이 맞은 비율'**이 '있음→없음' 수정의 정확도, **'Qwen 호소가 맞은 비율'**이 '없음→있음' 수정의 정확도에 가깝다. 0.9보다 낮으면 03에서 그 증상을 '삭제만'으로 낮추는 것을 검토한다.
(그 칸들은 예전 모델이 틀린 어려운 칸이라 실제보다 낮게 나올 수 있다)

In [ ]:
agree = agree_with_claude(judged)
if agree is not None:
    agree.to_csv(f"{OUT}/Claude_일치율.csv", encoding="utf-8-sig")
    display(agree)

## 5. 검수 파일
`검수_val.xlsx`(표)와 `검수_val.txt`(녹취록 포함)를 만든다. 처리 종류별로 뽑는다: 없음으로 수정 40, 있음으로 수정 40, 삭제 30, 그대로 40.
txt를 Claude에게 주면 칸마다 O/X로 확인해 준다(약 15만 토큰). 사람도 50칸 정도 엑셀에 O/X를 적는다.
O/X를 채운 엑셀은 03에서 `cleaning.review_accuracy`로 처리 종류 × 증상별 정확도를 본다.
0.9보다 낮은 증상은 03의 `MODES`에서 '삭제만'(수정 대신 삭제) 또는 '안 함'(원래 라벨)으로 바꾼다.

In [ ]:
sample = review_sample(judged)
sample.to_excel(f"{OUT}/검수_val.xlsx", index=False)
dialogues = dict(zip(va_cells["대화ID"], va_cells["대화"]))
write_review_txt(sample, dialogues, f"{OUT}/검수_val.txt")
print(sample["처리"].value_counts().to_string())

In [ ]:
# Train 판정 시간 예상 (블록 시간이 저장돼 있지 않으면 이번 실행 시간 기준)
n_tr = 0
try:
    tr_df, _ = load_labels()
    n_tr = len(build_cells(tr_df))
except Exception as e:
    print(e)
if n_tr:
    print(f"\nTrain {n_tr:,}칸 ≈ Val의 {n_tr / len(va_cells):.1f}배 → 02 예상 약 {val_min * n_tr / len(va_cells):.0f}분"
          " (01을 한 번에 끝까지 돌린 경우에만 정확)")